In [1]:
from OLS import OLS
import pandas as pd
import statsmodels.api as sm

In [2]:
# Load data from statsmodels
data = sm.datasets.macrodata.load_pandas()
df = data.data
display(df.head())

,year,quarter,realgdp,realcons,realinv,realgovt,realdpi,cpi,m1,tbilrate,unemp,pop,infl,realint
0,1959.0,1.0,2710.349,1707.4,286.898,470.045,1886.9,28.98,139.7,2.82,5.8,177.146,0.00,0.00
1,1959.0,2.0,2778.801,1733.7,310.859,481.301,1919.7,29.15,141.7,3.08,5.1,177.830,2.34,0.74
2,1959.0,3.0,2775.488,1751.8,289.226,491.260,1916.4,29.35,140.5,3.82,5.3,178.657,2.74,1.09
3,1959.0,4.0,2785.204,1753.7,299.356,484.052,1931.3,29.37,140.0,4.33,5.6,179.386,0.27,4.06
4,1960.0,1.0,2847.699,1770.5,331.722,462.199,1955.5,29.54,139.6,3.50,5.2,180.007,2.31,1.19


In [3]:
# Define target and predictor variables
y = df["infl"]
X = df[['realgdp',
 'realcons',
 'realinv',
 'realgovt',
 'realdpi',
 'cpi',
 'm1',
 'unemp',
 'pop',
 'realint']]

In [4]:
# Fit statsmodels regression
ols_sm = sm.OLS(y, sm.add_constant(X))
ols_sm = ols_sm.fit()

print(ols_sm.summary())

                            OLS Regression Results                            
Dep. Variable:                   infl   R-squared:                       0.775
Model:                            OLS   Adj. R-squared:                  0.764
Method:                 Least Squares   F-statistic:                     66.21
Date:                Wed, 25 Feb 2026   Prob (F-statistic):           8.45e-57
Time:                        15:57:31   Log-Likelihood:                -375.51
No. Observations:                 203   AIC:                             773.0
Df Residuals:                     192   BIC:                             809.5
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -31.4718     10.136     -3.105      0.0

In [5]:
# Fit and check coefficients
ols = OLS(add_intercept=True)
ols.fit(X, y)

result_df = pd.DataFrame([ols.beta_hat,
                          ols.beta_var**(0.5),
                          ols.beta_hat_tstat,
                          ols.beta_hat_pvals], columns=["const"]+X.columns.tolist(), index=["coef", "std err", "t", "P>|t|"]).T

print(result_df.round(3))
print("\n")

print(f"R-squared: {ols.r2:.3f}")
print(f"Adj. R-squared: {ols.r2adj:.3f}")
print(f"F-statistic: {ols.f_stat:.2f}")
print(f"Prob (F-statistic): {ols.f_stat_pval:.2f}")
print(f"Log-Likelihood: {ols.log_likelihood:.2f}")
print(f"AIC: {ols.aic:.2f}")
print(f"BIC: {ols.bic:.2f}")

            coef  std err       t  P>|t|
const    -31.472   10.136  -3.105  0.002
realgdp   -0.000    0.003  -0.040  0.968
realcons  -0.009    0.003  -2.741  0.007
realinv   -0.000    0.002  -0.231  0.818
realgovt   0.001    0.002   0.216  0.829
realdpi    0.006    0.002   2.442  0.016
cpi        0.148    0.027   5.397  0.000
m1        -0.023    0.002 -10.620  0.000
unemp     -0.452    0.218  -2.076  0.039
pop        0.225    0.074   3.041  0.003
realint   -0.965    0.054 -17.929  0.000


R-squared: 0.775
Adj. R-squared: 0.764
F-statistic: 66.21
Prob (F-statistic): 0.00
Log-Likelihood: -375.51
AIC: 773.03
BIC: 809.47


In [6]:
# Linearity Check
lin_check = ols.check_linearity(y=y, X=X)
print(lin_check)

           Residuals       infl
stat                           
coef    6.801124e-13   1.000000
tstat   3.068513e-11  45.117734
pvalue  1.000000e+00   0.000000
r2      4.086963e-24   0.775211


In [7]:
# Normality Check
norm_check = ols.check_normality(y=y, X=X)

Jarque–Bera Test
JB statistic : 24.6867
p-value      : 0.000004
Skewness     : 0.7834
Kurtosis     : 3.6808


In [8]:
# Outliers Check
outlier_check = ols.check_outliers()

print(outlier_check.sort_values("cooks_distance", ascending=False).head())

     leverage  resid_studentized  cooks_distance
197  0.161721          -1.807560        0.057302
88   0.063552           2.907814        0.052166
202  0.196402           1.498415        0.049886
89   0.039215           3.434109        0.043759
186  0.122950           1.753710        0.039194


In [9]:
# Multicollinearity Check
corr_mat, cond_num, vifs = ols.check_multicollinearity(X)

Variance Inflation Factors:
realgdp     8373.481777
realcons    4579.027343
realdpi     2456.252668
pop          615.498354
cpi          228.246780
realinv      103.555538
m1            79.551299
realgovt       9.490823
unemp          8.150892
realint        1.666832
Name: vif, dtype: float64


In [10]:
# Homoscedasticity Check
bp_test, resids_std_lin_check = ols.check_homoscedasticity(X=X)
print(resids_std_lin_check)

Breusch-Pagan Test
LM statistic : 29.1095
LM p-value      : 0.001196
F statistic     : 3.2141
F p-value     : 0.0008
        Studentized Residual
stat                        
coef               -0.000004
tstat              -0.000308
pvalue              0.999755
r2                  0.000006


In [11]:
# Autocorrelation Check
acf_stats, pacf_stats = ols.check_autocorrelation(y=y, X=X)